In [0]:
file_path ='/Volumes/workspace/default/sales_details'
df = (spark.read.format("csv")
      .option("header", "true")
      .option("inferSchema", "true")
      .load(file_path))

df.limit(2).toPandas()



,transaction_yr,transaction_mth,transaction_wk,transaction_dt,transaction_status,transaction_source,live_response_code,payment_instrument,bank_code,hyp_flag,asp_flag,merchant_id,merchant_status,marketplace_id,pg_id,flipkart_emi_flag,marketplace_context,is_shopsy_order,emi_flag,adonc_flag,count_of_tx,acct_cnt,tot_amt,tot_eff_amt_paymnt
0,2026,4,14,2026-04-01,FAILED,iOSApp,None,DEBIT,KOTAK,Others,<500,mp_flipkart,INVALIDATED,None,None,No,FLIPKART,None,None,No adonc,1,1,329,329
1,2026,4,15,2026-04-09,FAILED,iOSApp,None,UPI_COLLECT,PAYTM,Others,10k-20k,mp_flipkart,INVALIDATED,None,None,No,HYPERLOCAL,None,None,No adonc,1,1,12127,12127


In [0]:
from pyspark.sql.functions import col, hash, current_timestamp
clean_df = df.fillna({
    "tot_amt":0.0,
    "tot_eff_amt_paymnt": 0.0,
    "count_of_tx": 0,
    "transaction_status": "Unknown",
    "payment_instrument": "Unknown",
    "bank_code": "Unknown"
})
clean_df = clean_df.withColumn(
    "payment_profile_id",
    hash(col("payment_instrument"), col("bank_code"), col("pg_id"), col("emi_flag"))
    )
dim_date = clean_df.select("transaction_dt", "transaction_yr", "transaction_mth", "transaction_wk").distinct()
dim_merchant = clean_df.select("merchant_id", "merchant_status", "marketplace_id", "marketplace_context", "is_shopsy_order").distinct()
dim_payment = clean_df.select("payment_profile_id", "payment_instrument", "bank_code", "pg_id", "emi_flag", "flipkart_emi_flag").distinct()

fact_transaction = clean_df.select(
    "transaction_dt", 
    "merchant_id", 
    "payment_profile_id",
    "transaction_status",
    "live_response_code",
    "count_of_tx",
    "tot_amt",
    "tot_eff_amt_paymnt"
).withColumn("etl_loaded_at", current_timestamp())
dim_date.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_date")
dim_merchant.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_merchant")
dim_payment.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_payment")
fact_transaction.write.format("delta").mode("overwrite").saveAsTable("workspace.default.fact_transactions")
print("Pipeline Ready")


Pipeline Ready


In [0]:
%sql
select
p.payment_instrument,
m.merchant_status,
count(f.transaction_dt) as total_transaction_count,
sum(f.tot_amt) as total_processed,
sum(f.tot_eff_amt_paymnt) as total_effective_amt

from workspace.default.fact_transactions f
join workspace.default.dim_merchant m
on f.merchant_id = m.merchant_id
join workspace.default.dim_payment p 
on p.payment_profile_id = f.payment_profile_id
group by p.payment_instrument,
m.merchant_status
order by total_transaction_count DESC



payment_instrument,merchant_status,total_transaction_count,total_processed,total_effective_amt
CREDIT,CAPTURED,2766511,303230991420,289629087799
CREDIT,INVALIDATED,2343065,256739880322,245224496963
CREDIT,SUCCESS,2129857,233444018272,222971895044
DEBIT,CAPTURED,2018707,65539939531,64326234324
DEBIT,INVALIDATED,1709817,55475647513,54448666184
CREDIT,PENDING,1707896,187003213675,178617610709
DEBIT,SUCCESS,1548175,50391954749,49458335359
CREDIT,FAILED,1496225,163758496960,156416154125
DEBIT,PENDING,1246482,40379209486,39632313974
DEBIT,FAILED,1092082,35347827981,34694294408
